# Sequential Score Matching via SMC Temperature Bridging — LJ13

Trains a $\beta$-conditioned EGNN score network on the 13-particle
Lennard-Jones cluster (LJ13, 39-dimensional) using an SMC-discovered
temperature ladder. All operations are projected to the zero centre-of-mass
subspace throughout (see Section 1 for why this is still exact under the
*individual* harmonic trap used here).

Run top-to-bottom after a kernel restart. Set `resume = True` to continue
from a checkpoint.

## Imports and Seeds

In [1]:
from __future__ import annotations

import time
import pickle
from dataclasses import dataclass
from functools import partial
from pathlib import Path

import numpy as np
import jax
import jax.numpy as jnp
from jax.scipy.special import logsumexp
import optax
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from jax_pdf import LennardJones

from progressive_is_tempering.net.egnn_jax import EGNNDynamics
from progressive_is_tempering.sde import ou_forward_sample
from progressive_is_tempering.resampling import systematic_resample, normalized_ess
from progressive_is_tempering.mppt_rejuvenate import mppt_rejuvenate
from progressive_is_tempering.buffer import ReplayBuffer
from progressive_is_tempering.utils import (
    C_INK,
    C_INK_2,
    C_TEST,
    plot_value_comparison,
    print_seed_quality_report,
    smc_seed_quality_figure,
    style_axis,
)

SEED = 0
key = jax.random.key(SEED)

print(f"JAX devices: {jax.devices()}")
print(f"Default backend: {jax.default_backend()}")

JAX devices: [CudaDevice(id=0), CudaDevice(id=1), CudaDevice(id=2), CudaDevice(id=3)]
Default backend: gpu


## 0. Configuration

In [2]:
@dataclass
class ModelConfig:
    """EGNN architecture hyperparameters."""
    n_particles: int = 13
    spatial_dim: int = 3
    x_dim: int = 39          # n_particles * spatial_dim
    hidden_nf: int = 128
    n_layers: int = 5
    recurrent: bool = True
    attention: bool = True
    tanh: bool = True
    condition_time: bool = True
    condition_beta: bool = True
    beta_embedding_dim: int = 8
    coords_range: float = 15.0


@dataclass
class TrainingConfig:
    """All training / sampling hyperparameters (LJ13-specific values)."""
    # SDE
    T: float = 4.0
    t_min: float = 0.01
    n_sde_steps_train: int = 25
    n_sde_steps_eval: int = 1000

    # Outer loop
    num_outer_steps: int = 50
    n_particles: int = 5_000     # K: particles per backward SDE run

    # Inner loop
    n_inner_steps: int = 100
    batch_size: int = 512        # B per temperature

    # MALA rejuvenation (mppt's own SMC-tuned kernel; see mppt_rejuvenate)
    n_mala_rejuv: int = 2_000       # MALA steps per particle per outer step

    # Clipping (LJ13: the r^-12 repulsion makes ∇E blow up for close pairs;
    # clip PER PARTICLE — reshape to (n_particles, spatial_dim) and clamp each
    # particle's own displacement norm — rather than the whole flat-vector
    # norm used for DW4. A single close contact should only damp that pair's
    # target, not rescale all 13 particles uniformly. See diagnostic cell in
    # Section 13 for the empirical grad-norm distribution behind this default.
    score_clip: float = 1e4
    target_clip_norm: float = 150.0   # per-particle max norm (not global)

    # Optimiser
    learning_rate: float = 1e-3
    grad_clip: float = 1.0

    # MPPT SMC (temperature ladder) — ESS ratio and MCMC steps mirror
    # mppt/notebooks/LJ13.ipynb; the population no longer does (see below).
    #
    # One number governs the SMC population, the per-rung seed, and the buffer
    # capacity: they are the same particles at three points in their life. SMC
    # propagates N particles up the ladder, each rung's block is fed whole into
    # that rung's buffer (LJ13 already did this — it never subsampled), and the
    # outer loop evicts them gradually (FIFO) as backward-SDE data arrives.
    #
    # Raised from the reference notebook's 10k to 50k so that buffer capacity is
    # unchanged at 50k. The seed drains in n_smc_particles / n_particles = 10
    # outer steps, 20% of num_outer_steps.
    n_smc_particles: int = 25_000
    mppt_local_ess_ratio: float = 0.2
    mppt_n_mcmc_steps: int = 2_000

    # Checkpointing / logging
    log_every: int = 1
    save_every: int = 1
    eval_every: int = 10
    n_eval_particles: int = 20_000

    # --- Derived: seed size and buffer capacity are the SMC population -------
    @property
    def buffer_capacity(self) -> int:
        return self.n_smc_particles

    @property
    def n_seed_particles(self) -> int:
        return self.n_smc_particles

    @property
    def seed_flush_steps(self) -> float:
        """Outer steps until FIFO has fully retired the SMC seed."""
        return self.n_smc_particles / self.n_particles


model_cfg = ModelConfig()
train_cfg = TrainingConfig()

# The seed fills every buffer to capacity, so FIFO needs `seed_flush_steps`
# outer steps to retire it. If that exceeds num_outer_steps the run finishes
# while still training partly on SMC particles, which are far more degenerate
# than their row count suggests (see the seed-quality diagnostic).
_flush = train_cfg.seed_flush_steps
print(f"buffer: {train_cfg.buffer_capacity:,}/rung "
      f"= {_flush:.3g} generations at K={train_cfg.n_particles:,}")
print(f"seed retired after outer step {_flush:.3g} "
      f"of {train_cfg.num_outer_steps} "
      f"({_flush / train_cfg.num_outer_steps:.0%} of the run)")
if _flush >= train_cfg.num_outer_steps:
    print("  WARNING: seed never fully drains — raise num_outer_steps, "
          "or lower n_smc_particles")

resume: bool = False

NOTEBOOK_NAME = "lj13"
CHECKPOINT_DIR = Path("../../checkpoints") / NOTEBOOK_NAME
LOG_DIR = Path("../../logging")
FIGURES_DIR = Path("../../figures") / NOTEBOOK_NAME
EVAL_FIGURES_DIR = FIGURES_DIR / "eval"
for d in [CHECKPOINT_DIR, LOG_DIR, FIGURES_DIR, EVAL_FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = CHECKPOINT_DIR / "checkpoint.pkl"
LOG_PATH = LOG_DIR / f"{NOTEBOOK_NAME}.txt"

buffer: 25,000/rung = 5 generations at K=5,000
seed retired after outer step 5 of 50 (10% of the run)


## 1. Target Distribution (LJ13)

Thirteen particles in 3D, flat representation $x \in \mathbb{R}^{39}$.
Energy:
$$U(x) = \underbrace{\sum_{i<j}\varepsilon\Bigl[(r_m/r_{ij})^{12}-2(r_m/r_{ij})^6\Bigr]}_{\text{LJ pair potential}}
+ \underbrace{\tfrac{s}{2}\sum_i \|x_i\|^2}_{\text{individual harmonic trap}}$$
with $\varepsilon=1$, $r_m=1$, $s=1$ (`trap_scale=1.0`, `trap_mode='individual'`),
matching `mppt/notebooks/LJ13.ipynb` and `pita`'s LJ13 config.

**Why we can still work on the zero-CoM subspace.** Unlike DW4's
`trap_scale=0`, the individual trap here is *not* translation-invariant on
its face. But by the parallel-axis theorem,
$$\sum_i \|x_i\|^2 = \sum_i \|x_i - \mathrm{CoM}\|^2 + N\|\mathrm{CoM}\|^2,$$
so $U$ splits into a CoM-only Gaussian term plus a term depending purely on
the CoM-zero (relative) coordinates. Evaluated at any CoM-zero point the
$N\|\mathrm{CoM}\|^2$ term vanishes identically, so `energy(x)` restricted
to CoM-zero points is exactly (up to an additive constant) the unnormalized
log-density of the internal-coordinate marginal — the same role `trap_scale=0`
plays for DW4. This is why every `project_com` call, and the EGNN's
hard-wired zero-CoM output, transfer unchanged from `dw4.py`.

In [3]:
lj13 = LennardJones(n_particles=13, spatial_dim=3, epsilon=2.0, rm=1.0,
                     trap_scale=1.0, trap_mode='individual', beta=1.0)

# Local override of LennardJones.__call__'s pairwise-distance floor: jax_pdf
# hard-codes sqrt(diff**2 + 1e-8), which barely softens the r^-12 term for
# near-collision pairs (only matters below r ~ sqrt(eps)). We widen the
# floor to 1e-5 (r ~ 0.0032) to further dampen this blowup during
# early/off-equilibrium sampling, without editing the shared jax_pdf package
# (used by other projects in this env). Params below mirror `lj13` above.
_R_EPS = 1e-5


def _lj13_logpdf(x: jnp.ndarray) -> jnp.ndarray:
    """log π(x) for LJ13, matching LennardJones.__call__ but with _R_EPS floor."""
    pos = x.reshape(model_cfg.n_particles, model_cfg.spatial_dim)
    diff = pos[:, None, :] - pos[None, :, :]
    r = jnp.sqrt(jnp.sum(diff ** 2, axis=-1) + _R_EPS)
    i, j = jnp.triu_indices(model_cfg.n_particles, k=1)
    r_pairs = r[i, j]

    inv_r = lj13.rm / r_pairs
    u_lj = jnp.sum(lj13.epsilon * (inv_r ** 12 - 2.0 * inv_r ** 6))
    u_trap = 0.5 * lj13.trap_scale * jnp.sum(pos ** 2)
    return -lj13.beta * (u_lj + u_trap)


def energy(x: jnp.ndarray) -> jnp.ndarray:
    """E(x) = -log π(x) + const.  Expects flat (39,) input."""
    return -_lj13_logpdf(x)


# Quick sanity check. Unlike DW4's quartic potential, LJ's 1/r^12 term is
# singular when all particles coincide, so x=0 gives E=inf, grad=nan here —
# use a random point instead (still huge: randomly scattered particles are
# far from the equilibrium cluster geometry the LJ potential favours).
key, subkey = jax.random.split(key)
x_test = jax.random.normal(subkey, (model_cfg.x_dim,))
print(f"LJ13 dim: {lj13.dim}")
print(f"energy(random) = {energy(x_test):.4e}")
print(f"∇E norm at random point = {jnp.linalg.norm(jax.grad(energy)(x_test)):.4e}")

LJ13 dim: 39
energy(random) = 5.2467e+01
∇E norm at random point = 9.4416e+02


## 2. Geometry Utilities, Reference Data, and Comparison Plots

All particle states are kept in the zero-CoM subspace throughout.

This section also holds the ground-truth reference samples and the two
summary statistics every comparison in this notebook uses — **pairwise
distances** (cluster geometry) and **energy** (how physically plausible a
configuration is). They live here, ahead of SMC, because three separate
places compare against them: the SMC seed diagnostic (Section 9), the
periodic eval during training (Section 14), and the final eval (Section 16).
One definition, one styling, so the three are directly comparable.

In [4]:
def project_com(x: jnp.ndarray) -> jnp.ndarray:
    """Project flat (39,) or batched (..., 39) to zero centre-of-mass.

    Subtracts the per-sample mean particle position in each spatial direction.
    Works for any leading batch dimensions.
    """
    shape = x.shape
    xr = x.reshape(-1, model_cfg.n_particles, model_cfg.spatial_dim)
    com = xr.mean(axis=-2, keepdims=True)   # (..., 1, D)
    return (xr - com).reshape(shape)


# Verify projection
key, subkey = jax.random.split(key)
_xr = jax.random.normal(subkey, (5, model_cfg.x_dim))
_xp = jax.vmap(project_com)(_xr)
_com = _xp.reshape(5, model_cfg.n_particles, model_cfg.spatial_dim).mean(axis=1)
print(f"Max |CoM| after projection: {jnp.abs(_com).max():.2e}  (should be ≈ 0)")

Max |CoM| after projection: 6.88e-08  (should be ≈ 0)


In [5]:
# ---- Summary statistics -----------------------------------------------------
@jax.jit
def get_pairwise_distances(x_flat: jnp.ndarray) -> jnp.ndarray:
    """All 78 unique pairwise distances for (B, 39) input → (B, 78)."""
    pos = x_flat.reshape(-1, model_cfg.n_particles, model_cfg.spatial_dim)   # (B, 13, 3)
    diff = pos[:, :, None, :] - pos[:, None, :, :]                            # (B, 13, 13, 3)
    r = jnp.sqrt(jnp.sum(diff ** 2, axis=-1) + 1e-8)                         # (B, 13, 13)
    i, j = jnp.triu_indices(model_cfg.n_particles, k=1)                       # 78 unique pairs
    return r[:, i, j]                                                          # (B, 78)


@jax.jit
def get_energies(x_flat: jnp.ndarray) -> jnp.ndarray:
    """Per-sample energy E(x) for (B, 39) input → (B,)."""
    return jax.vmap(energy)(x_flat.reshape(-1, model_cfg.x_dim))


# ---- Reference (ground-truth) samples ---------------------------------------
# Held-out LJ13 test split, already projected to zero CoM (max|CoM| ≈ 4e-7).
_pt_ref = jnp.array(np.load(Path("../../data/test_split_LJ13-1000.npy")))  # (10000, 39)
_ref_com = _pt_ref.reshape(-1, model_cfg.n_particles, model_cfg.spatial_dim).mean(axis=1)
print(f"Loaded reference samples: {_pt_ref.shape}  "
      f"max|CoM|={jnp.abs(_ref_com).max():.2e}  (should be ≈ 0)")

_REF_R = get_pairwise_distances(_pt_ref)      # (10000, 78)
_REF_E = get_energies(_pt_ref)                 # (10000,)
print(f"Reference energy: mean={float(_REF_E.mean()):.2f}  "
      f"[min={float(_REF_E.min()):.2f}, max={float(_REF_E.max()):.2f}]")

# ---- Comparison plot helpers ------------------------------------------------
# Two series everywhere: the thing under test vs. the ground-truth reference.
# The styling, the reference-anchored window, and the honest-density histogram
# live in `progressive_is_tempering/utils/smc_diagnostics.py` (imported as
# `style_axis` / `plot_value_comparison` above) so that all three notebooks
# render directly comparable figures. Colour follows the entity, never the
# panel — C_TEST is always whatever we are evaluating (SMC seed, or the model),
# C_REF is always ground truth — and the reference is additionally dashed, so
# identity never rests on colour alone.
_style_axis = style_axis   # local alias: this notebook styles its own axes too


def plot_distance_comparison(ax, r_test, label_test, r_ref=_REF_R, bins=150):
    """Pairwise-distance density: test vs reference, on shared bin edges."""
    return plot_value_comparison(
        ax, r_test, r_ref, "inter-particle distance $r_{ij}$",
        label_test, bins=bins, clip_at_zero=True,   # distances are non-negative
    )


def plot_energy_comparison(ax, e_test, label_test, e_ref=_REF_E, bins=150):
    """Energy density: test vs reference, on shared bin edges.

    Energy is the sharper of the two statistics: it weights close contacts by
    r^-12, so a single bad pair moves it by orders of magnitude while barely
    perturbing the distance histogram. The off-window fraction reported here is
    the real early-training diagnostic.
    """
    return plot_value_comparison(
        ax, e_test, e_ref, "energy $E(x)$", label_test, bins=bins,
    )

Loaded reference samples: (10000, 39)  max|CoM|=4.22e-07  (should be ≈ 0)
Reference energy: mean=-43.13  [min=-66.18, max=-10.21]


## 3. MPPT Interface for LJ13

The geometric path bridges
$\rho(x) = \mathcal{N}(x;0,I)$ (restricted to CoM=0)
to $\pi(x) \propto e^{-E(x)}$ via
$\pi_\beta(x) \propto \exp\{-(1-\beta)\tfrac12\|x\|^2 - \beta E(x)\}$.

In [6]:
def logpdf_ref(x: jnp.ndarray) -> jnp.ndarray:
    """log N(x; 0, I) up to a constant (reference on the CoM-zero subspace)."""
    return -0.5 * jnp.sum(x ** 2)


def logpdf_final_target(x: jnp.ndarray) -> jnp.ndarray:
    """log π(x) = -E(x) (unnormalized, LJ13 log-density)."""
    return -energy(x)


def rvs_ref(key: jnp.ndarray) -> jnp.ndarray:
    """Draw one sample from N(0, I) projected to zero CoM."""
    return project_com(jax.random.normal(key, (model_cfg.x_dim,)))

In [7]:
# mppt_problem is built here (rather than at the SMC call in Section 9) so
# Section 7's MALA demo can also use it, and mppt's kernel-construction
# helpers (needed there too) are imported once, in one place.
import sys
sys.path.insert(0, str(Path("../../mppt").resolve()))

from mppt import PTProblem, smc_sampler

mppt_problem = PTProblem(
    rvs_ref=rvs_ref,
    logpdf_ref=logpdf_ref,
    logpdf_final_target=logpdf_final_target,
)

## 4. Forward Process

OU SDE with closed-form marginals:
$X_t \mid X_0 \sim \mathcal{N}(e^{-t}X_0, \sigma_t^2 I)$,
$\sigma_t^2 = 1 - e^{-2t}$.
After noising we project back to zero CoM.

In [8]:
@jax.jit
def ou_forward_sample_com(key: jnp.ndarray, x0: jnp.ndarray, t: float) -> jnp.ndarray:
    """OU noising of a single sample, projected to zero CoM."""
    xt = ou_forward_sample(key, x0, t)
    return project_com(xt)

## 5. Score Network (EGNN)

$\beta$-conditioned $E(n)$-equivariant GNN dynamics network
$s_\theta(x_t, t, \beta)$ approximating the excess score
$\nabla_{x_t}\log(p_t^\beta / p_t^\rho)$.

Input: flat coordinates $x_t \in \mathbb{R}^{39}$, time $t$, temperature $\beta$.
Interface: **batched** — `(B, 39)`, `(B, 1)`, `(B, 1)` → `(B, 39)`.
The EGNN output is automatically zero-CoM (velocity = displacement from input,
CoM-projected inside `EGNNDynamics`).

In [9]:
score_net = EGNNDynamics(
    n_particles=model_cfg.n_particles,
    spatial_dim=model_cfg.spatial_dim,
    hidden_nf=model_cfg.hidden_nf,
    n_layers=model_cfg.n_layers,
    recurrent=model_cfg.recurrent,
    attention=model_cfg.attention,
    tanh=model_cfg.tanh,
    condition_time=model_cfg.condition_time,
    condition_beta=model_cfg.condition_beta,
    beta_embedding_dim=model_cfg.beta_embedding_dim,
    coords_range=model_cfg.coords_range,
)

key, subkey = jax.random.split(key)
_x_init = jnp.zeros((1, model_cfg.x_dim))
_t_init = jnp.zeros((1, 1))
_b_init = jnp.zeros((1, 1))
params = score_net.init(subkey, _x_init, _t_init, _b_init)
n_params = sum(x.size for x in jax.tree_util.tree_leaves(params))
print(f"EGNN initialized: {n_params:,} parameters")
print(f"n_particles={model_cfg.n_particles}, spatial_dim={model_cfg.spatial_dim}, "
      f"hidden_nf={model_cfg.hidden_nf}, n_layers={model_cfg.n_layers}")

EGNN initialized: 580,485 parameters
n_particles=13, spatial_dim=3, hidden_nf=128, n_layers=5


## 6. CoM-Projected Backward SDE

The backward SDE time-reversal of the OU process, with CoM projection applied
after each exponential-integrator step. Both the forward OU density
$p_\rho(x_n \mid x_{n-1})$ and the backward proposal
$q_\theta(x_{n-1} \mid x_n)$ are evaluated at the projected state.

The control function wraps the EGNN (batched interface) with a batch
dimension of 1 for single-particle calls inside vmap.

In [10]:
def backward_sde_batch_com(
    key: jnp.ndarray,
    control_fn,
    beta: jnp.ndarray,
    energy_fn,
    dim: int,
    n_particles: int,
    T: float,
    n_steps: int,
) -> tuple[jnp.ndarray, jnp.ndarray]:
    """Batched backward SDE: all n_particles processed together in one scan.

    control_fn(x, t, beta) receives x of shape (n_particles, dim) and returns
    the same shape — one EGNN call per time step rather than one per particle.

    Returns x0 (K, dim) and path-space log-weights (K,).
    """
    dt = T / n_steps
    times = jnp.linspace(T, dt, n_steps)

    lam = jnp.exp(-dt)
    sigma_sq = 1.0 - jnp.exp(-2.0 * dt)

    key, subkey = jax.random.split(key)
    x_T = project_com(jax.random.normal(subkey, (n_particles, dim)))  # (K, dim)
    init_log_w = 0.5 * jnp.sum(x_T ** 2, axis=-1)                    # (K,)

    def step_fn(carry, t_n):
        x_n, log_w, rng = carry  # x_n: (K, dim), log_w: (K,)
        rng, subkey = jax.random.split(rng)

        control = control_fn(x_n, t_n, beta)              # (K, dim)
        eps = jax.random.normal(subkey, x_n.shape)         # (K, dim)

        # Exponential-integrator backward step, then project
        mu_bwd = lam * x_n + (1.0 - lam) * control        # (K, dim)
        x_prev = mu_bwd + jnp.sqrt(sigma_sq) * eps         # (K, dim)
        x_prev = project_com(x_prev)                        # (K, dim)

        # Sum over dim axis to get per-particle scalar log-densities
        log_p = -0.5 * jnp.sum((x_n - lam * x_prev) ** 2, axis=-1) / sigma_sq  # (K,)
        log_q = -0.5 * jnp.sum((x_prev - mu_bwd) ** 2, axis=-1) / sigma_sq      # (K,)

        return (x_prev, log_w + log_p - log_q, rng), None

    (x0, log_w, _), _ = jax.lax.scan(step_fn, (x_T, init_log_w, key), times)

    # Terminal correction: log π_β(x_0) for each particle
    energies = jax.vmap(energy_fn)(x0)                                       # (K,)
    log_w = log_w - (1.0 - beta) * 0.5 * jnp.sum(x0 ** 2, axis=-1) - beta * energies
    return x0, log_w


@partial(jax.jit, static_argnums=(3, 4, 5, 6, 7))
def _run_backward_sde(key, params, beta, energy_fn, dim, n_particles, T, n_steps):
    """JIT-compiled CoM-projected backward SDE runner."""
    score_clip = train_cfg.score_clip

    def control(x, t, b):
        # x: (K, dim) — full particle batch; one EGNN call per SDE step
        K = x.shape[0]
        s = score_net.apply(
            params,
            x,                                  # (K, dim)
            jnp.full((K, 1), t),               # (K, 1)
            jnp.full((K, 1), b),               # (K, 1)
        )                                       # (K, dim)
        if score_clip > 0:
            s = jnp.clip(s, -score_clip, score_clip)
        return 2.0 * s

    return backward_sde_batch_com(key, control, beta, energy_fn, dim, n_particles, T, n_steps)


# Quick sanity check with random parameters
key, subkey = jax.random.split(key)
_x0_demo, _lw_demo = _run_backward_sde(
    subkey, params, jnp.array(1.0), energy,
    model_cfg.x_dim, 50, train_cfg.T, train_cfg.n_sde_steps_train,
)
_path_ess = normalized_ess(_lw_demo)
print(f"Backward SDE demo (β=1, 50 particles, 100 steps):")
print(f"  log_w range: [{float(_lw_demo.min()):.2f}, {float(_lw_demo.max()):.2f}]")
print(f"  Normalized ESS: {float(_path_ess):.3f}  (1.0=perfect)")
_com_demo = _x0_demo.reshape(-1, model_cfg.n_particles, model_cfg.spatial_dim).mean(axis=1)
print(f"  Max |CoM| in samples: {jnp.abs(_com_demo).max():.2e}  (should be ≈ 0)")

Backward SDE demo (β=1, 50 particles, 100 steps):
  log_w range: [-7846501548032.00, -63163.69]
  Normalized ESS: 0.020  (1.0=perfect)
  Max |CoM| in samples: 3.67e-07  (should be ≈ 0)


## 7. CoM-Projected MALA Rejuvenation

Rejuvenation uses mppt's own MALA-mode HMC kernel (`mppt_rejuvenate`, from
`progressive_is_tempering/mppt_rejuvenate.py`) — the *same* kernel SMC
(Section 9) tunes per rung via `keep_mcmc_params=True` — rather than a
separately-calibrated implementation. mppt's kernel has no CoM-awareness at
all, so `mppt_rejuvenate`'s `project_fn` hook re-projects to zero-CoM after
*every* internal move (never leaving the constraint manifold), matching the
safety margin the retired `mala_step_com` provided.

This demo runs before SMC has produced a real ladder, so it calibrates a
throwaway kernel directly via mppt's own `adapt_params` (not a bisection
reimplementation) at an illustrative rung — this is *not* the calibration
used for training; that comes from SMC itself in Section 9.

In [11]:
from mppt.abstract.mcmc import batch_init_extend as _batch_init_extend_demo
from mppt.impl.pt_mcmc_hmc import hmc_pt_mcmc as _hmc_pt_mcmc_demo

_demo_beta, _demo_t_idx = jnp.array(0.1), 1
_demo_mcmc = _hmc_pt_mcmc_demo(mppt_problem, _demo_beta, _demo_t_idx, mala=True)

key, subkey = jax.random.split(key)
_pts = jax.vmap(project_com)(jax.random.normal(subkey, (20, model_cfg.x_dim)))
key, k_init, k_adapt, k_move = jax.random.split(key, 4)
_demo_x = _batch_init_extend_demo(k_init, _pts, _demo_mcmc)
_demo_params = _demo_mcmc.adapt_params(k_adapt, _demo_x)

_pts_mala, _mala_acc = mppt_rejuvenate(
    k_move, _pts, mppt_problem, _demo_beta, _demo_t_idx, _demo_params, 5,
    project_fn=project_com,
)
print(f"MALA demo (β={float(_demo_beta):.3f}, rung 1, 5 steps): "
      f"acceptance rate = {float(_mala_acc):.3f}")

MALA demo (β=0.100, rung 1, 5 steps): acceptance rate = 1.000


## 8. IS Bridging Between Adjacent Temperatures

$$\log w_{k \to k+1}(x) = (\beta_{k+1}-\beta_k)\bigl(-E(x)+\tfrac12\|x\|^2\bigr)$$

In [12]:
@jax.jit
def is_bridge_log_weights(samples, beta_k, beta_next):
    """IS log-weights to bridge π_{β_k} → π_{β_{k+1}}.

    Reference definition of the formula above, kept for interactive diagnostics.
    The training path does not call it: `_data_gen_parallel` (Section 12)
    recomputes it inline so the whole pipeline stays inside one jit region.
    """
    delta_beta = beta_next - beta_k
    energies = jax.vmap(energy)(samples)
    norms_sq = jnp.sum(samples ** 2, axis=-1)
    return delta_beta * (-energies + 0.5 * norms_sq)


@jax.jit
def normalize_log_weights(log_w: jnp.ndarray) -> jnp.ndarray:
    """Shift a block of log-weights to mean weight 1 (i.e. logsumexp = log n).

    A buffer holds several blocks written at different times, and `.sample()`
    resamples across all of them jointly — so the blocks' weight *scales* must
    be commensurable. Raw bridge weights Δβ(-E + ½‖x‖²) run O(10-100) for LJ13,
    whereas the SMC seed particles are already ≈ π_{β_k} and so carry weight 0.
    Mixing those two raw would let one block dominate the buffer outright,
    depending only on sign. Normalizing every block on the way in makes blocks
    combine in proportion to their particle counts, which is what we want.

    Degenerate blocks (all weights -inf) fall back to uniform.
    """
    n = log_w.shape[0]
    total = logsumexp(log_w)
    return jnp.where(
        jnp.isfinite(total),
        log_w - total + jnp.log(n),
        jnp.zeros_like(log_w),
    )

## 9. Temperature Ladder via SMC (MPPT)

Run SMC once before training to discover $\{\beta_k\}_{k=0}^M$ such that
successive IS bridges have normalized ESS $\geq$ `local_ess_ratio`. The SMC
engine is MALA-mode (matching `mppt_rejuvenate`'s own kernel, below).

`keep_all_states=True` returns the particle positions at **every** rung,
stacked as `(len(betas_list), n_smc_particles, 39)` and indexed in the same
order as `betas_list`. Unlike DW4/GMM we keep these: they are the only
samples in the whole script that are actually distributed according to
$\pi_{\beta_k}$ at each level, and are used to seed the replay buffers
(Section 11).

`keep_mcmc_params=True` additionally returns the per-rung MALA kernel
(covariance + step size) SMC itself adapted while building the ladder. That
*same* kernel — not a separately recalibrated one — is reused for
outer-loop rejuvenation (Section 12); see `mppt_rejuvenate`. The buffer
seed (Section 11) uses SMC's particle output directly, with no further
MALA pass.

**No parallel-tempering refinement.** Earlier revisions followed SMC with a
single `pt_manual_runners` call. That call could not affect the run: the
ladder comes from SMC, and the returned `pt_state` was never read — so it
only paid the eager-JIT cost of the PT step (10-60s) for nothing. Particle
diversification is now done explicitly, with MALA, in the outer loop
(Section 12).

In [13]:
# mppt_problem and the sys.path/import setup were done in Section 3.

# Early-load checkpoint before SMC/PT/calibration so we can skip them on resume
_ckpt_data = None
if resume and CHECKPOINT_PATH.exists():
    with open(CHECKPOINT_PATH, "rb") as f:
        _ckpt_data = pickle.load(f)
    print(f"Checkpoint found: outer_step={_ckpt_data['outer_step']}, "
          f"will restore temperature ladder, buffers, and the MALA kernel")

if _ckpt_data is None:
    key, subkey = jax.random.split(key)
    print("Running SMC to discover temperature ladder …")
    t0 = time.time()
    smc_particles, logZ_dict, betas_list, mcmc_params_by_rung = smc_sampler(
        subkey,
        mppt_problem,
        n_particles=train_cfg.n_smc_particles,
        engine={"name": "mala"},
        n_mcmc_steps=train_cfg.mppt_n_mcmc_steps,
        local_ess_ratio=train_cfg.mppt_local_ess_ratio,
        verbose=True,
        runtime_check=False,
        keep_all_states=True,
        keep_mcmc_params=True,
    )
    print(f"SMC done in {time.time()-t0:.1f}s")
    print(f"Temperature ladder ({len(betas_list)} levels): {[f'{b:.3f}' for b in betas_list]}")
    print(f"logZ estimates: {logZ_dict}")

    # Scalar MALA step size (ε), adapted per rung by SMC's own acceptance-rate
    # bisection (HMCParameters.scaled_cov.eps) — the same value reused wherever
    # mcmc_params_by_rung is indexed below.
    _eps_by_rung = mcmc_params_by_rung.scaled_cov.eps
    print("MALA step size (ε) adapted by SMC, per rung:")
    for _k, _eps_k in enumerate(_eps_by_rung, start=1):
        print(f"  rung {_k} (β={betas_list[_k]:.3f}): ε={float(_eps_k):.5g}")

    betas = jnp.array(betas_list)
    M = len(betas_list) - 1
    print(f"M={M} non-reference temperature levels")

    # Project the SMC particles onto the zero-CoM subspace. MPPT's MCMC engines
    # do not project (nothing in mppt/impl/mcmc_*.py touches the CoM), and the
    # individual trap only *confines* the CoM rather than pinning it — the CoM
    # random-walks with std ~ 1/sqrt(N·s·β) per direction. Everything downstream
    # of here assumes CoM-zero states, so project once, at the boundary.
    smc_states = jax.vmap(jax.vmap(project_com))(smc_particles)  # (n_levels, N, 39)
    assert smc_states.shape[0] == len(betas_list), (
        f"expected one particle block per rung, got {smc_states.shape[0]} "
        f"blocks for {len(betas_list)} rungs"
    )
    _smc_com = smc_states.reshape(-1, model_cfg.n_particles, model_cfg.spatial_dim).mean(axis=1)
    print(f"SMC particles: {smc_states.shape}  max|CoM| after projection: "
          f"{jnp.abs(_smc_com).max():.2e}  (should be ≈ 0)")
else:
    betas_list = _ckpt_data["betas_list"]
    betas = jnp.array(betas_list)
    M = len(betas_list) - 1
    mcmc_params_by_rung = _ckpt_data["mcmc_params_by_rung"]
    print(f"Restored temperature ladder ({len(betas_list)} levels): {[f'{b:.3f}' for b in betas_list]}")
    print(f"M={M} non-reference temperature levels")

Running SMC to discover temperature ladder …
Initializing and compiling...
--- SMC anneal | estimators: (SMC_like, reverse_logistic, robust) ---
    dlogZ = log(Z_t / Z_prev)   |   logZ(t) = log(Z_t / Z_0)
iter   1 | t=0.001953 | acc=0.64 | niter=2000 | dlogZ=(   -1.845,    -1.847,    -1.848) | logZ(t)=(   -1.845,    -1.847,    -1.848)
iter   2 | t=0.103317 | acc=0.52 | niter=2000 | dlogZ=(   -0.111,    -0.120,    -0.128) | logZ(t)=(   -1.956,    -1.968,    -1.976)
iter   3 | t=0.316980 | acc=0.56 | niter=2000 | dlogZ=(    3.591,     3.601,     3.598) | logZ(t)=(    1.635,     1.633,     1.622)
iter   4 | t=0.498407 | acc=0.53 | niter=2000 | dlogZ=(    4.503,     4.488,     4.490) | logZ(t)=(    6.138,     6.121,     6.112)
iter   5 | t=0.682586 | acc=0.57 | niter=2000 | dlogZ=(    6.185,     6.183,     6.183) | logZ(t)=(   12.323,    12.304,    12.295)
iter   6 | t=0.885929 | acc=0.59 | niter=2000 | dlogZ=(    8.774,     8.790,     8.789) | logZ(t)=(   21.096,    21.094,    21.084)
it

### SMC seed quality at $\beta = 1$

The top rung of the SMC ladder targets $\pi_{\beta_M} = \pi$ — the same
distribution the reference set is drawn from — so the two should agree. This
is the one check worth running *before* committing to a training run: it puts
a ceiling on what the seeded buffers can teach the network. If SMC has not
reached the target at $\beta = 1$, no amount of training on its output will.

Two failure modes, and they are not the same thing:

1. **Distributional mismatch** (top row). Both statistics are shown because
   they fail differently. Pairwise distances describe cluster *geometry* and
   are forgiving — a badly-strained cluster can still produce a
   plausible-looking distance histogram. Energy is the sharp test: it weights
   close contacts by $r^{-12}$, so a single bad pair moves it by orders of
   magnitude. Agreement on distances but not energy means the geometry is
   roughly right and the fine structure is not.
2. **Ancestral degeneracy** (bottom row). The histograms match beautifully,
   yet the population carries far less independent information than its
   particle count suggests. SMC resampling collapses the ancestral lineages;
   local MCMC then re-equilibrates each particle *within* its basin, erasing
   literal duplicates while the basin assignment stays frozen. A distinct-row
   count reports 100% unique and misses this entirely — which matters here,
   because the training log shows top-rung buffer ESS collapsing to ~0.01.

The degeneracy test uses the 78 pairwise distances as exchangeable features:
particle-permutation symmetry means every pair must give the same exceedance
probability $P(r_{ij} > \tau)$. The reference set goes through the identical
computation as a control — being i.i.d., it should recover its own size, which
is what makes the test's verdict on the SMC block trustworthy. See
`progressive_is_tempering/utils/smc_diagnostics.py`.

Skipped on resume, where `smc_states` does not exist.

In [14]:
if _ckpt_data is None:
    _smc_top = smc_states[-1]                      # (N, 39) at β = β_M = 1
    _r_smc = get_pairwise_distances(_smc_top)
    _e_smc = get_energies(_smc_top)

    fig, _seed_stats = smc_seed_quality_figure(
        features_test=np.asarray(_r_smc),
        features_ref=np.asarray(_REF_R),
        energy_test=np.asarray(_e_smc),
        energy_ref=np.asarray(_REF_E),
        feature_label="inter-particle distance $r_{ij}$",
        title=(f"SMC seed quality at β={float(betas[-1]):.3f} vs ground truth "
               f"({_smc_top.shape[0]:,} particles)"),
        features_nonneg=True,
    )
    fig.savefig(FIGURES_DIR / "lj13_smc_seed_quality.png", dpi=150)
    plt.close(fig)

    print(f"SMC seed at β={float(betas[-1]):.3f}:")
    print(f"  energy    mean={float(_e_smc.mean()):10.2f}   "
          f"reference mean={float(_REF_E.mean()):10.2f}")
    print(f"  energy  median={float(jnp.median(_e_smc)):10.2f}   "
          f"reference median={float(jnp.median(_REF_E)):10.2f}")
    print(f"  distance  mean={float(_r_smc.mean()):10.4f}   "
          f"reference mean={float(_REF_R.mean()):10.4f}")
    print_seed_quality_report(_seed_stats)
    print(f"  → {FIGURES_DIR / 'lj13_smc_seed_quality.png'}")
else:
    print("Resumed from checkpoint — SMC seed diagnostic skipped (no smc_states).")

SMC seed at β=1.000:
  energy    mean=    -42.93   reference mean=    -43.13
  energy  median=    -43.35   reference median=    -43.72
  distance  mean=    1.6421   reference mean=    1.6391
  SMC seed: 25,000 particles, 78 exchangeable features
  threshold tau                 : 1.6092
  pooled P(feature > tau)       : 0.5016   reference 0.5000
  s.e. if 25,000 i.i.d. particles : 0.00316
  observed spread across features: 0.00358  (1x the i.i.d. s.e.)
  implied independent particles : 19,546 of 25,000
    control — same statistic on the reference: 1,453 of 10,000
  chi2(78) = 119, p = 1.90e-03
  off-window fraction: feature 0.00%, energy 0.01%
  → ../../figures/lj13/lj13_smc_seed_quality.png


## 11. Per-Temperature Replay Buffers (Seeded from SMC)

Buffer 0 ($\beta=0$): i.i.d. $\mathcal{N}(0,I)$ projected to zero CoM, frozen.
The reference *is* the $\beta=0$ target, so exact draws beat SMC particles here.

Buffers $1,\ldots,M$: seeded directly with this rung's SMC particles — the
particles SMC itself already produced after its own reweight/resample/MALA
move at that rung (Section 9), with no further rejuvenation applied here.
Buffers $2,\ldots,M$ additionally receive backward-SDE data during training,
and FIFO eviction retires the seed once ~`buffer_capacity` new samples have
arrived.

Seed particles are already $\approx \pi_{\beta_k}$, so they enter with
**uniform** log-weights — no bridge weights (those are for particles arriving
from the rung below). See `normalize_log_weights` in Section 8 for why every
block is normalized on the way in.

In [15]:
if _ckpt_data is None:
    buffers: list[ReplayBuffer] = [
        ReplayBuffer(capacity=train_cfg.buffer_capacity, dim=model_cfg.x_dim)
        for _ in range(len(betas_list))
    ]

    # Buffer 0: exact projected N(0, I), frozen. Filled to capacity rather than
    # to n_particles: it is never evicted, and exact draws are free, so leaving
    # slots empty would only narrow the β=0 training data for no reason.
    key, subkey = jax.random.split(key)
    buf0_samples = jax.vmap(project_com)(
        jax.random.normal(subkey, (train_cfg.buffer_capacity, model_cfg.x_dim))
    )
    buffers[0].add(buf0_samples, jnp.zeros(train_cfg.buffer_capacity))
    buffers[0].freeze()
    print(f"Buffer 0 (β=0): {buffers[0].size} samples, uniform weights [frozen]")

    # Buffers 1..M: the whole SMC block at the matching rung — SMC's own
    # reweight/resample/MALA move (Section 9) already produced these, already
    # projected to zero CoM. The block is exactly buffer capacity, so each
    # buffer starts full and the seed drains over seed_flush_steps outer steps.
    assert smc_states.shape[1] == train_cfg.buffer_capacity, (
        f"SMC block {smc_states.shape[1]:,} must equal capacity "
        f"{train_cfg.buffer_capacity:,}; ReplayBuffer.add would otherwise write "
        f"duplicate indices and silently corrupt"
    )
    print("Seeding buffers directly from SMC output …")
    t0 = time.time()
    for _k in range(1, len(betas_list)):
        _seed_k = smc_states[_k]
        buffers[_k].add(_seed_k, jnp.zeros(_seed_k.shape[0]))
        _frozen = " [frozen]" if _k == 1 else ""
        print(f"  Buffer {_k} (β={float(betas[_k]):.3f}): {buffers[_k].size} samples{_frozen}")

    # Buffer 1 is never written during training (the outer loop fills 2..M), so
    # it stays frozen — but it now holds real π_{β_1} particles rather than
    # buffer 0 reweighted, which is why its ESS starts at 1.0 instead of ~0.3.
    if len(betas_list) >= 2:
        buffers[1].freeze()
    print(f"Seeding done in {time.time()-t0:.1f}s")
    print(f"Buffers 2..{len(betas_list)-1} additionally receive backward-SDE data "
          f"during training.")
else:
    buffers = _ckpt_data["buffers"]
    print(f"Restored {len(buffers)} replay buffers from checkpoint")
    for _k, _buf in enumerate(buffers):
        print(f"  Buffer {_k} (β={float(betas[_k]):.3f}): {_buf.size} samples")

Buffer 0 (β=0): 25000 samples, uniform weights [frozen]
Seeding buffers directly from SMC output …
  Buffer 1 (β=0.002): 25000 samples [frozen]
  Buffer 2 (β=0.103): 25000 samples
  Buffer 3 (β=0.317): 25000 samples
  Buffer 4 (β=0.498): 25000 samples
  Buffer 5 (β=0.683): 25000 samples
  Buffer 6 (β=0.886): 25000 samples
  Buffer 7 (β=1.000): 25000 samples
Seeding done in 0.1s
Buffers 2..7 additionally receive backward-SDE data during training.


## 12. Parallel Data Generation (Outer Loop)

For each active temperature level $k \in \{1,\ldots,M-1\}$ **in parallel**:
1. Backward SDE at $\beta_k$ → $(x_0^{(i)}, \log w_\mathrm{path}^{(i)})$
2. Systematic resample → approx. i.i.d. from $\pi_{\beta_k}$
3. CoM-projected MALA rejuvenation
4. IS bridge weights → $\log w_{k \to k+1}$

In [16]:
@partial(jax.jit, static_argnums=(3, 4, 5, 6, 7, 8, 9))
def _data_gen_parallel(
    sde_keys,            # (M_active,) PRNG keys
    resample_keys,       # (M_active,) PRNG keys
    mala_keys,           # (M_active,) PRNG keys
    energy_fn,           # static callable
    dim,                 # static int
    n_particles,         # static int
    T,                   # static float
    n_sde_steps,         # static int
    n_mala_steps,        # static int
    mppt_problem,        # static -- mppt.PTProblem, same object SMC ran with
    params,              # network pytree
    betas_active,        # (M_active,)
    betas_next,          # (M_active,)
    t_idx_active,        # (M_active,) rung indices k = 1 … M-1, matching betas_active
    mcmc_params_active,  # HMCParameters pytree (leading axis M_active), SMC-tuned per rung
):
    """Run full data-generation pipeline for all active temperature levels via vmap."""
    score_clip = train_cfg.score_clip

    def process_one(sde_key, resample_key, mala_key, beta_k, beta_next,
                     t_idx_k, mcmc_params_k):
        def control(x, t, b):
            # x: (K, dim) — full particle batch; one EGNN call per SDE step
            K = x.shape[0]
            s = score_net.apply(
                params,
                x,                              # (K, dim)
                jnp.full((K, 1), t),           # (K, 1)
                jnp.full((K, 1), b),           # (K, 1)
            )                                   # (K, dim)
            if score_clip > 0:
                s = jnp.clip(s, -score_clip, score_clip)
            return 2.0 * s

        # 1. Backward SDE
        x0_batch, log_w_path = backward_sde_batch_com(
            sde_key, control, beta_k, energy_fn, dim, n_particles, T, n_sde_steps
        )
        # 2. Systematic resample
        resample_idx = systematic_resample(resample_key, log_w_path, n_particles)
        x0_resampled = x0_batch[resample_idx]
        # 3. MALA rejuvenation, using the same SMC-tuned kernel from Section 9
        x0_rejuv, mala_acc = mppt_rejuvenate(
            mala_key, x0_resampled, mppt_problem, beta_k, t_idx_k,
            mcmc_params_k, n_mala_steps, project_fn=project_com,
        )
        # 4. IS bridge weights
        delta_beta = beta_next - beta_k
        energies = jax.vmap(energy_fn)(x0_rejuv)
        norms_sq = jnp.sum(x0_rejuv ** 2, axis=-1)
        bridge_lw = delta_beta * (-energies + 0.5 * norms_sq)
        path_ess = normalized_ess(log_w_path)
        return x0_rejuv, bridge_lw, path_ess, mala_acc

    return jax.vmap(process_one)(
        sde_keys, resample_keys, mala_keys, betas_active, betas_next,
        t_idx_active, mcmc_params_active,
    )

## 13. Training Step (Inner Loop)

DDPM regression target for LJ13:
$$\phi(x_0,t,\beta) = e^{-t}\,\beta\,(x_0 - \nabla E(x_0))$$
EGNN is called on the full batch `(M*B, 39)` directly (no per-sample vmap).
Forward-noised samples are projected to zero CoM before passing to the network.

Unlike DW4, the target is clipped **per particle** (reshape to
`(n_particles, spatial_dim)`, clamp each particle's own displacement norm)
rather than by the whole flat-vector norm — see the diagnostic below for why.

In [17]:
def clip_per_particle_norm(x_flat: jnp.ndarray, max_norm: float) -> jnp.ndarray:
    """Clamp each particle's own vector norm to max_norm, direction preserved.

    x_flat: (..., n_particles * spatial_dim). A single close contact only
    damps that particle pair's own displacement, unlike a whole-vector clip
    which would rescale all particles uniformly.
    """
    shape = x_flat.shape
    xr = x_flat.reshape(-1, model_cfg.n_particles, model_cfg.spatial_dim)
    norms = jnp.linalg.norm(xr, axis=-1, keepdims=True)
    clip_coeff = jnp.minimum(max_norm / (norms + 1e-6), 1.0)
    return (xr * clip_coeff).reshape(shape)


# Diagnostic: empirical per-particle |∇E| distribution, equilibrium vs.
# off-equilibrium (random Gaussian — representative of early backward-SDE /
# low-β configurations, where a chance close pair can make ∇E blow up by many
# orders of magnitude). Sanity-checks the target_clip_norm default above.
_grad_fn_demo = jax.vmap(jax.grad(energy))
key, subkey = jax.random.split(key)
_gt_probe = jax.vmap(project_com)(jax.random.normal(subkey, (2000, model_cfg.x_dim)))
_n_probe = jnp.linalg.norm(
    _grad_fn_demo(_gt_probe).reshape(-1, model_cfg.n_particles, model_cfg.spatial_dim), axis=-1
)
print(f"Per-particle |∇E| on random CoM-zero configs: "
      f"mean={_n_probe.mean():.2f}  p99={jnp.percentile(_n_probe, 99):.2f}  "
      f"max={_n_probe.max():.2e}  (target_clip_norm={train_cfg.target_clip_norm})")

Per-particle |∇E| on random CoM-zero configs: mean=inf  p99=13486071808.00  max=inf  (target_clip_norm=150.0)


In [18]:
optimizer = optax.chain(
    optax.clip_by_global_norm(train_cfg.grad_clip),
    optax.adam(train_cfg.learning_rate),
)
opt_state = optimizer.init(params)


@jax.jit
def train_step(params, opt_state, x0_all, t_all, beta_all, key):
    """Loss, gradient, and Adam update.

    x0_all: (M*B, 39), t_all: (M*B,), beta_all: (M*B,)
    """
    target_clip = train_cfg.target_clip_norm
    score_clip = train_cfg.score_clip

    def loss_fn(p):
        B_total = x0_all.shape[0]

        # Forward OU noising, then project to zero CoM
        noise_keys = jax.random.split(key, B_total)
        sigma_t = jnp.sqrt(1.0 - jnp.exp(-2.0 * t_all))
        means = jnp.exp(-t_all[:, None]) * x0_all
        eps = jax.vmap(jax.random.normal, in_axes=(0, None))(noise_keys, (x0_all.shape[1],))
        xt_all = means + sigma_t[:, None] * eps
        xt_all = jax.vmap(project_com)(xt_all)  # project to zero CoM

        # DDPM targets (stop_gradient: no backprop through energy Hessian)
        def phi_one(x0_i, t_i, beta_i):
            lambda_t = jnp.exp(-t_i)
            grad_e = jax.grad(energy)(x0_i)
            phi = lambda_t * beta_i * (x0_i - grad_e)
            # Per-particle norm clipping
            if target_clip > 0:
                phi = clip_per_particle_norm(phi, target_clip)
            return phi

        targets = jax.lax.stop_gradient(
            jax.vmap(phi_one)(x0_all, t_all, beta_all)
        )

        # EGNN predictions — fully batched
        preds = score_net.apply(p, xt_all, t_all[:, None], beta_all[:, None])
        if score_clip > 0:
            preds = jnp.clip(preds, -score_clip, score_clip)

        return jnp.mean(jnp.sum((preds - targets) ** 2, axis=-1))

    loss, grads = jax.value_and_grad(loss_fn)(params)
    updates, opt_state_new = optimizer.update(grads, opt_state)
    params_new = optax.apply_updates(params, updates)
    return params_new, opt_state_new, loss

## 14. Periodic Evaluation

Same three candidate sample sets and two summary statistics as the final
evaluation (Section 16) — pairwise distances and energy, for raw backward-SDE
output, IS-resampled backward-SDE output, and buffer[M] — run every
`eval_every` outer steps so the training-time figures are directly comparable
to the end-of-run one (minus its loss-curve column).

Energy is the statistic that moves first. Early in training the model puts
mass on configurations with close contacts, whose $r^{-12}$ energies are
enormous; that shows up as a large off-window fraction long before the
distance histogram looks obviously wrong.

In [19]:
def periodic_eval_plot(params, outer_step):
    """Same three candidate sample sets and two summary statistics as the final
    evaluation (Section 16) — distances (top row) and energy (bottom row) for
    raw backward-SDE output, IS-resampled backward-SDE output, and buffer[M] —
    minus the loss-curve column, which belongs to the end-of-run figure only.
    """
    global key

    key, subkey = jax.random.split(key)
    x0_eval, lw_eval = _run_backward_sde(
        subkey, params, betas[-1], energy,
        model_cfg.x_dim, train_cfg.n_eval_particles,
        train_cfg.T, train_cfg.n_sde_steps_eval,
    )
    ess_eval = float(normalized_ess(lw_eval))

    # (1) Raw backward-SDE output: the network alone, no correction.
    x_raw = x0_eval

    # (2) Same particles, IS-corrected by the path-space weights.
    key, subkey = jax.random.split(key)
    resamp_idx = systematic_resample(subkey, lw_eval, len(lw_eval))
    x_resamp = x0_eval[resamp_idx]

    # (3) buffer[M]: the training data at β=1, drawn the way the inner loop
    # draws it — systematic resampling on the stored IS weights.
    if buffers[M].is_ready(1):
        key, subkey = jax.random.split(key)
        n_buf = min(train_cfg.n_eval_particles, buffers[M].size)
        x_buf, _ = buffers[M].sample(subkey, n_buf)
        buf_ess = float(normalized_ess(buffers[M].log_weights[: buffers[M].size]))
    else:
        x_buf, buf_ess = None, float("nan")

    sample_sets = [
        ("Backward SDE (raw, no IS)", "SDE raw", x_raw,
         f"unweighted, {x_raw.shape[0]:,} particles"),
        ("Backward SDE + IS resample", "SDE + IS", x_resamp,
         f"path ESS={ess_eval:.3f}  (≈{ess_eval * x_raw.shape[0]:,.0f} eff.)"),
        (f"buffer[M]  β={float(betas[M]):.3f}", "buffer[M]", x_buf,
         f"buffer ESS={buf_ess:.3f}" if x_buf is not None else "empty"),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    for _col, (_label, _short, _x, _sub) in enumerate(sample_sets):
        ax_r, ax_e = axes[0, _col], axes[1, _col]
        if _x is None:
            for _ax in (ax_r, ax_e):
                _ax.text(0.5, 0.5, "buffer[M] empty", ha="center", va="center",
                         transform=_ax.transAxes, color=C_INK_2, fontsize=10)
                _ax.set_axis_off()
            continue
        _r = get_pairwise_distances(_x)
        _e = get_energies(_x)
        plot_distance_comparison(ax_r, _r, _short)
        plot_energy_comparison(ax_e, _e, _short)
        ax_r.set_title(f"{_label}\n{_sub}", color=C_INK, fontsize=10)
        ax_e.set_title(f"Energy — {_short}", color=C_INK, fontsize=10)

    fig.suptitle(
        f"Step {outer_step+1}: three candidate sample sets vs ground truth  "
        f"(path ESS={ess_eval:.3f})",
        fontsize=12, color=C_INK,
    )
    plt.tight_layout()
    fname = EVAL_FIGURES_DIR / f"eval_step{outer_step+1:04d}.png"
    fig.savefig(fname, dpi=150)
    plt.close(fig)
    print(f"  [eval] step {outer_step+1}  model ESS={ess_eval:.3f}  → {fname}")

## 15. Training Loop

```
for outer_step in 1 … num_outer_steps:
    for k = 1, …, M−1:  (parallel via jax.vmap)
        CoM-projected backward SDE at β_k  →  (x₀, log w_path)
        systematic resample  →  approx. i.i.d. from π_{β_k}
        CoM-projected MALA rejuvenation
        IS bridge weights    →  log w_{k→k+1}
        buffer[k+1].add(x₀, log w_{k→k+1})
    for inner_step in 1 … n_inner_steps:
        sample mini-batches from each buffer
        project-noised xt, compute DDPM targets
        Adam step on joint loss via batched EGNN
```

In [20]:
start_outer_step = 0
loss_history = []

if _ckpt_data is not None:
    params = _ckpt_data["params"]
    opt_state = _ckpt_data["opt_state"]
    start_outer_step = _ckpt_data["outer_step"]
    loss_history = _ckpt_data.get("loss_history", [])
    key = jax.random.wrap_key_data(_ckpt_data["key"])
    print(f"Resumed from outer step {start_outer_step}")
else:
    if LOG_PATH.exists():
        with open(LOG_PATH) as f:
            n_attempts = f.read().count("Attempt")
    else:
        n_attempts = 0
    with open(LOG_PATH, "a") as f:
        f.write(f"\nAttempt {n_attempts + 1}\n")
    print(f"Starting fresh (Attempt {n_attempts + 1})")

Starting fresh (Attempt 8)


In [21]:
print(f"\nStarting training: outer steps {start_outer_step} → {train_cfg.num_outer_steps}")
train_start = time.time()

for outer_step in range(start_outer_step, train_cfg.num_outer_steps):
    step_start = time.time()

    # ---- Outer loop: parallel SMC data generation ----
    M_active = len(betas_list) - 2   # levels k = 1 … M-1
    outer_ess_list = []

    if M_active > 0:
        all_keys = jax.random.split(key, 3 * M_active + 1)
        key = all_keys[0]
        sde_keys    = all_keys[1            : M_active + 1]
        resamp_keys = all_keys[M_active + 1 : 2 * M_active + 1]
        mala_keys   = all_keys[2 * M_active + 1:]

        # Active rungs are k = 1 … M-1 (M_active of them); mcmc_params_by_rung[k-1]
        # is the HMCParameters SMC tuned for rung k (Section 9), so slice its
        # first M_active entries to match betas[1:-1].
        t_idx_active = jnp.arange(1, M)
        mcmc_params_active = jax.tree_util.tree_map(
            lambda leaf: leaf[:M_active], mcmc_params_by_rung
        )

        x0_rejuv_all, bridge_lw_all, path_ess_all, _ = _data_gen_parallel(
            sde_keys, resamp_keys, mala_keys,
            energy,
            model_cfg.x_dim, train_cfg.n_particles,
            train_cfg.T, train_cfg.n_sde_steps_train,
            train_cfg.n_mala_rejuv,
            mppt_problem,
            params, betas[1:-1], betas[2:],
            t_idx_active, mcmc_params_active,
        )

        outer_ess_list = path_ess_all.tolist()

        for i in range(M_active):
            # Normalize each block on the way in so it is commensurable with
            # the uniform-weight SMC seed already in the buffer, and with
            # blocks written at earlier outer steps (Section 8).
            buffers[i + 2].add(x0_rejuv_all[i], normalize_log_weights(bridge_lw_all[i]))

    # ---- Inner loop: gradient descent ----
    all_buffers_ready = all(
        buffers[k].is_ready(train_cfg.batch_size) for k in range(len(betas_list))
    )
    if all_buffers_ready:
        inner_loss_acc = jnp.zeros(())
        for inner_step in range(train_cfg.n_inner_steps):
            x0_batches, t_batches, beta_batches = [], [], []
            for k in range(len(betas_list)):
                key, subkey = jax.random.split(key)
                x0_k, _ = buffers[k].sample(subkey, train_cfg.batch_size)

                key, subkey = jax.random.split(key)
                t_k = jax.random.uniform(
                    subkey, (train_cfg.batch_size,),
                    minval=train_cfg.t_min, maxval=train_cfg.T,
                )
                x0_batches.append(x0_k)
                t_batches.append(t_k)
                beta_batches.append(jnp.full(train_cfg.batch_size, betas[k]))

            x0_all = jnp.concatenate(x0_batches, axis=0)
            t_all  = jnp.concatenate(t_batches,  axis=0)
            beta_all = jnp.concatenate(beta_batches, axis=0)

            key, subkey = jax.random.split(key)
            params, opt_state, loss_val = train_step(
                params, opt_state, x0_all, t_all, beta_all, subkey
            )
            inner_loss_acc = inner_loss_acc + loss_val

        mean_loss = float(inner_loss_acc / train_cfg.n_inner_steps)
        loss_history.append(mean_loss)
    else:
        mean_loss = float("nan")

    step_time = time.time() - step_start

    # ---- Logging ----
    if (outer_step + 1) % train_cfg.log_every == 0:
        ess_str = "  ".join(
            f"β{k+1}={ess:.3f}" for k, ess in enumerate(outer_ess_list)
        ) if outer_ess_list else "—"
        buf_ess_str = "  ".join(
            f"β{k}={float(normalized_ess(buffers[k].log_weights[:buffers[k].size])):.3f}"
            if buffers[k].size > 0 else f"β{k}=—"
            for k in range(len(betas_list))
        )
        elapsed = time.time() - train_start
        log_line = (
            f"[step {outer_step+1:4d}/{train_cfg.num_outer_steps}] "
            f"loss={mean_loss:.4f}  path_ESS=[{ess_str}]  buf_ESS=[{buf_ess_str}]  "
            f"step={step_time:.1f}s  elapsed={elapsed:.1f}s"
        )
        print(log_line)
        with open(LOG_PATH, "a") as f:
            f.write(log_line + "\n")

    # ---- Checkpoint ----
    if (outer_step + 1) % train_cfg.save_every == 0:
        ckpt = {
            "params": params,
            "opt_state": opt_state,
            "outer_step": outer_step + 1,
            "loss_history": loss_history,
            "key": jax.random.key_data(key),
            "betas_list": betas_list,
            "mcmc_params_by_rung": mcmc_params_by_rung,
            "buffers": buffers,
            "_pt_ref": _pt_ref,
        }
        with open(CHECKPOINT_PATH, "wb") as f:
            pickle.dump(ckpt, f)
        print(f"  Checkpoint saved at step {outer_step+1}")

    # ---- Periodic evaluation ----
    if (outer_step + 1) % train_cfg.eval_every == 0:
        periodic_eval_plot(params, outer_step)

print(f"\nTraining complete in {time.time()-train_start:.1f}s")


Starting training: outer steps 0 → 50
[step    1/50] loss=5792.2891  path_ESS=[β1=0.000  β2=0.000  β3=0.000  β4=0.000  β5=0.000  β6=0.000]  buf_ESS=[β0=1.000  β1=1.000  β2=0.566  β3=0.274  β4=0.589  β5=0.002  β6=1.000  β7=1.000]  step=84.0s  elapsed=84.3s
  Checkpoint saved at step 1
[step    2/50] loss=5764.7803  path_ESS=[β1=0.008  β2=0.002  β3=0.001  β4=0.000  β5=0.001  β6=0.000]  buf_ESS=[β0=1.000  β1=1.000  β2=0.392  β3=0.220  β4=0.379  β5=0.002  β6=0.480  β7=0.772]  step=51.9s  elapsed=136.3s
  Checkpoint saved at step 2
[step    3/50] loss=5739.6895  path_ESS=[β1=0.007  β2=0.002  β3=0.006  β4=0.000  β5=0.001  β6=0.000]  buf_ESS=[β0=1.000  β1=1.000  β2=0.299  β3=0.184  β4=0.300  β5=0.002  β6=0.308  β7=0.646]  step=52.0s  elapsed=188.4s
  Checkpoint saved at step 3
[step    4/50] loss=5705.6099  path_ESS=[β1=0.010  β2=0.008  β3=0.000  β4=0.001  β5=0.000  β6=0.000]  buf_ESS=[β0=1.000  β1=1.000  β2=0.243  β3=0.161  β4=0.247  β5=0.002  β6=0.243  β7=0.531]  step=52.0s  elapsed=240.5s

## 16. Final Evaluation

Run the backward SDE at $\beta=1$ and compare the model against the
ground-truth reference on pairwise distances and energy, alongside the
training loss curve.

The three panels answer three different questions and are deliberately not
combined: distances say whether the *geometry* is right, energy says whether
the *physics* is right, and the loss curve says whether the network converged
at all. Loss keeps its own log-scale axis in its own panel — it shares no
units with either statistic.

Reference samples were loaded once in Section 2; nothing to re-run here.

In [22]:
print("Running final evaluation …")
key, subkey = jax.random.split(key)
x0_eval, lw_eval = _run_backward_sde(
    subkey, params, betas[-1], energy,
    model_cfg.x_dim, train_cfg.n_eval_particles,
    train_cfg.T, train_cfg.n_sde_steps_eval,
)
ess_eval = float(normalized_ess(lw_eval))
print(f"Evaluation path-space ESS (β=1): {ess_eval:.3f}")

Running final evaluation …
Evaluation path-space ESS (β=1): 0.000


### Three candidate definitions of "our method's samples"

| # | Sample set | What it isolates |
|---|---|---|
| 1 | Backward SDE output, **no** reweighting | the learned score alone |
| 2 | Same run, IS-resampled on the path weights | score **+** path-space correction |
| 3 | `buffer[M]`, IS-resampled on its stored weights | the training data at $\beta=1$ |

(1) and (2) come from the **single** SDE run above — the same particles,
differing only in whether `log w_path` is applied — so any gap between them is
exactly the work the importance correction is doing. If they agree, the
network learned the target; if only (2) matches, the reweighting is carrying
the result and `ess_eval` says how few particles are really behind it.

(3) needs no SDE run: it is what the outer loop deposited at the top rung,
i.e. what the inner loop actually trained on. It bounds what the network could
have learned — a network cannot beat its own training data.

In [23]:
# (1) Raw backward-SDE output: the network alone, no correction.
x_raw = x0_eval

# (2) Same particles, IS-corrected by the path-space weights.
key, subkey = jax.random.split(key)
resamp_idx = systematic_resample(subkey, lw_eval, len(lw_eval))
x_resamp = x0_eval[resamp_idx]

# (3) buffer[M]: the training data at β=1, drawn the way the inner loop draws
# it — systematic resampling on the stored IS weights. Raw buffer contents
# would NOT be π-distributed: the blocks arriving from the outer loop are
# ≈ π_{β_{M-1}} particles carrying bridge weights to β_M, and only become ≈ π
# once those weights are applied. `.sample()` applies them.
if buffers[M].is_ready(1):
    key, subkey = jax.random.split(key)
    n_buf = min(train_cfg.n_eval_particles, buffers[M].size)
    x_buf, _ = buffers[M].sample(subkey, n_buf)
    buf_ess = float(normalized_ess(buffers[M].log_weights[: buffers[M].size]))
else:
    x_buf, buf_ess = None, float("nan")

# (long label for panel headers, short label for tables, samples, subtitle)
sample_sets = [
    ("Backward SDE (raw, no IS)", "SDE raw", x_raw,
     f"unweighted, {x_raw.shape[0]:,} particles"),
    ("Backward SDE + IS resample", "SDE + IS", x_resamp,
     f"path ESS={ess_eval:.3f}  (≈{ess_eval * x_raw.shape[0]:,.0f} eff.)"),
    (f"buffer[M]  β={float(betas[M]):.3f}", "buffer[M]", x_buf,
     f"buffer ESS={buf_ess:.3f}" if x_buf is not None else "empty"),
]

In [24]:
# Columns: the three candidate sample sets, plus the loss curve.
# Rows: the two summary statistics. Distances say whether the *geometry* is
# right, energy says whether the *physics* is right — energy is far the sharper
# test, since r^-12 punishes close contacts. Every panel carries the same
# dashed ground-truth reference, so the three sets are read against each other.
fig, axes = plt.subplots(2, 4, figsize=(21, 8))

_summary = []
for _col, (_label, _short, _x, _sub) in enumerate(sample_sets):
    ax_r, ax_e = axes[0, _col], axes[1, _col]
    if _x is None:
        for _ax in (ax_r, ax_e):
            _ax.text(0.5, 0.5, "buffer[M] empty", ha="center", va="center",
                     transform=_ax.transAxes, color=C_INK_2, fontsize=10)
            _ax.set_axis_off()
        continue
    _r = get_pairwise_distances(_x)
    _e = get_energies(_x)
    plot_distance_comparison(ax_r, _r, _short)
    _frac = plot_energy_comparison(ax_e, _e, _short)
    ax_r.set_title(f"{_label}\n{_sub}", color=C_INK, fontsize=10)
    ax_e.set_title(f"Energy — {_short}", color=C_INK, fontsize=10)
    _summary.append((_short, _e, _r, _frac))

# Training loss curve (own axis — shares no units with either statistic)
ax_loss = axes[0, 3]
if loss_history:
    ax_loss.plot(loss_history, color=C_TEST, linewidth=1.5)
    ax_loss.set_yscale("log")
    ax_loss.set_xlabel("outer step", color=C_INK_2, fontsize=10)
    ax_loss.set_ylabel("DDPM loss (sum over dim)", color=C_INK_2, fontsize=10)
else:
    ax_loss.text(0.5, 0.5, "No loss history", ha="center", va="center",
                 transform=ax_loss.transAxes, color=C_INK_2, fontsize=10)
ax_loss.set_title("Training loss", color=C_INK, fontsize=11)
_style_axis(ax_loss)
axes[1, 3].set_axis_off()

fig.suptitle(
    f"LJ13 final evaluation — β={float(betas[-1]):.3f}, "
    f"three candidate sample sets vs ground truth",
    fontsize=12, color=C_INK,
)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "lj13_distances_energy_and_loss.png", dpi=150)
plt.close(fig)
print(f"Figure saved to {FIGURES_DIR / 'lj13_distances_energy_and_loss.png'}")

Figure saved to ../../figures/lj13/lj13_distances_energy_and_loss.png


In [25]:
# Numbers behind the figure. Energy is the statistic to read first.
print(f"\n── Final samples: three candidate definitions vs ground truth ──")
for _lbl, _short, _x, _sub in sample_sets:
    if _x is not None:
        print(f"    {_short:<10} {_lbl:<30} {_x.shape[0]:>7,}×{_x.shape[1]}   {_sub}")
print()
print(f"  {'set':<12}{'E mean':>13}{'E median':>13}{'r mean':>11}{'off-window':>13}")
print("  " + "─" * 62)
for _short, _e, _r, _frac in _summary:
    print(f"  {_short:<12}{float(_e.mean()):>13.2f}{float(jnp.median(_e)):>13.2f}"
          f"{float(_r.mean()):>11.4f}{_frac:>12.2%}")
print("  " + "─" * 62)
print(f"  {'reference':<12}{float(_REF_E.mean()):>13.2f}"
      f"{float(jnp.median(_REF_E)):>13.2f}{float(_REF_R.mean()):>11.4f}{'—':>13}")
print()


── Final samples: three candidate definitions vs ground truth ──
    SDE raw    Backward SDE (raw, no IS)       20,000×39   unweighted, 20,000 particles
    SDE + IS   Backward SDE + IS resample      20,000×39   path ESS=0.000  (≈4 eff.)
    buffer[M]  buffer[M]  β=1.000              20,000×39   buffer ESS=0.628

  set                E mean     E median     r mean   off-window
  ──────────────────────────────────────────────────────────────
  SDE raw      119162768.00       113.90     1.8019      39.03%
  SDE + IS           -37.65       -40.19     1.7072       0.02%
  buffer[M]          -42.22       -42.78     1.6550       0.01%
  ──────────────────────────────────────────────────────────────
  reference          -43.13       -43.72     1.6391            —



In [26]:
# Per-coordinate statistics (should have zero CoM)
com_resamp = x_resamp.reshape(-1, model_cfg.n_particles, model_cfg.spatial_dim).mean(axis=1)
print(f"\nCoM statistics of model samples:")
print(f"  Mean |CoM|: {jnp.abs(com_resamp).mean():.2e}  (should be ≈ 0)")

# Temperature ladder summary — one series, so one colour and no legend box;
# the title names it. Values are direct-labelled since there are only a handful.
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(len(betas_list)), betas_list, color=C_TEST, width=0.6)
for _k, _b in enumerate(betas_list):
    ax.text(_k, float(_b), f"{float(_b):.3f}", ha="center", va="bottom",
            fontsize=8, color=C_INK_2)
ax.set_xlabel("temperature level $k$", color=C_INK_2, fontsize=10)
ax.set_ylabel("$\\beta_k$", color=C_INK_2, fontsize=10)
ax.set_title(f"Temperature ladder ({len(betas_list)} levels)", color=C_INK, fontsize=11)
ax.set_ylim(0, 1.12)
_style_axis(ax)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "lj13_temperature_ladder.png", dpi=150)
plt.close(fig)
print(f"Temperature ladder figure saved.")
print(f"Ladder: {[f'{b:.3f}' for b in betas_list]}")


CoM statistics of model samples:
  Mean |CoM|: 8.97e-09  (should be ≈ 0)
Temperature ladder figure saved.
Ladder: ['0.000', '0.002', '0.103', '0.317', '0.498', '0.683', '0.886', '1.000']
